# HMAC Implementation 1
manually only with hashlib

In [ ]:
import hashlib

def HMAC_With_sha512(key: bytes, message: bytes) -> str:
   
    block_size = 128  #>>>>>>>>>>>>> we choosed 128 block size for key >>> SHA-512 >>>(128*8 = 1024)
    if len(key) > block_size:      #>>>>> if Key was larger than 128 
        key = hashlib.sha512(key).digest()  #>>>>> hash it with Sha-512
    #>>>>>>>>>>>> If it was smnaller than our key size -> padding method 
    if len(key) < block_size:
        key += b'\x00' * (block_size - len(key))  ## b -> binary form
    #>>>>>>>>>>> inner hash and outter hash 
    ipad = b'\x36' * block_size
    opad = b'\x5c' * block_size

    #>>>>>>>>>>>>>>>>> XOR the key with ipad
    key_ipad = bytes(a ^ b for a, b in zip(key, ipad))
    #>>>>>>>>>>>>>>>>XOR the key with opad
    key_opad = bytes(a ^ b for a, b in zip(key, opad))

    #>>>>>>>>>>>> inner hash = SHA512(key_ipad || plian text)
    inner = hashlib.sha512(key_ipad + message).digest()
    #>>>>>>>>>>>>>>> outer hash = SHA512(key_opad || inner)
    outer = hashlib.sha512(key_opad + inner).digest()

    return outer.hex()  ##>>>> string of hexadecimal format of the last hash(outer)

#####>>>>>>>>>> Test
key = b"jeffrey_epstein"   #>>>> input has to convert into binary format with b
message = b"I love eating barbeque bacon burger "

result = HMAC_With_sha512(key, message)
print("HMAC with SHA512 Result :")
print(result)

HMAC with SHA512 Result :
7d24e9a32dffd0512ccc6b1ad26692d915df75f02bb8bac3cc02adeaed4e4dbaa4a8a1c7c79102301d34550d33533de06e607425888068ac5228b572031cae7b


# HMAC Implementation 2
hashlib + hmac 


In [1]:
import hmac
import hashlib

key = b"onion_knight"
message = b"chicken macaroni and pizza"

#>>>>>>>>> with hmac lib 
###>>>>> SHA 512
h1 = hmac.new(key, message, hashlib.sha512)
print("HMAC with SHA512 :")
print(h1.hexdigest()) 
##>>>>>>>> SHA 256
h2 = hmac.new(key, message, hashlib.sha256)
print("HMAC with SHA256 :")
print(h2.hexdigest())
###>>>>>>>>>>> MD5
h3 = hmac.new(key, message, hashlib.md5)
print("HMAC with MD5 :")
print(h3.hexdigest())
#>>>>>>>>>>>>  SHA3 224
h4 = hmac.new(key, message, hashlib.sha3_224)
print("HMAC with SHA3(224) :")
print(h4.hexdigest())
print(h4.digest())

HMAC with SHA512 :
fbea180894d06e5af285610a59bcef0fa63619a2b39a1c950643a61c5044ea9cb07a65292e08e63fd57e6d6767f83bcf68bb4445070c03a730aed5d0840d2659
HMAC with SHA256 :
b0a45ae0d117c4236259ee28304d64340f3af3c9a0693101e90763be8630460f
HMAC with MD5 :
2642bb3cd73133ff29036fb0fd146481
HMAC with SHA3(224) :
31ca2166079e87c69cd0785be6a6c96abbc0756cd5f95b1d6f96682e
b'1\xca!f\x07\x9e\x87\xc6\x9c\xd0x[\xe6\xa6\xc9j\xbb\xc0ul\xd5\xf9[\x1do\x96h.'


# CMAC Implementation 1
the bad way

In [2]:
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.backends import default_backend

def xor_bytes(a: bytes, b: bytes) -> bytes:
                                   ########>>>>  2 bytes as input
    return bytes(x ^ y for x, y in zip(a, b))  ###>>> 1 bytes as output

def LEFT_shift(block: bytes) -> bytes:
    """shift left for 128 bits block"""
    shifted = int.from_bytes(block, 'big') << 1
    shifted &= (1 << 128) - 1
    return shifted.to_bytes(16, 'big')

#######>>>>>>>>>>>>>> AES for cmac
def cmac_aes_manual(key: bytes, message: bytes) -> str:
    """ Manual AES """
    if len(key) not in (16, 24, 32):  ##### in AES key has to be 16 or 24 or 32
        raise ValueError("Key must be 16, 24, or 32 bytes")

    #>>>>>>>>>>>>>>>>>>>   ciphering 
    cipher = Cipher(algorithms.AES(key), modes.ECB(), backend=default_backend())
    encryptor = cipher.encryptor()

    block_size = 16  #>>>>>>>>>>> AES block size = 128 bit

    #>>>>>>>>>>>>>>> create the Subkeys .........->>>>>>> (K1, K2)
    L = encryptor.update(b'\x00' * block_size) + encryptor.finalize()
    
    #>>>>>>>>>>>>>>>>>> Rb = 0^120 10000111
    Rb = b'\x00' * 15 + b'\x87'
    
    K1 = LEFT_shift(L)
    if L[0] & 0x80:   ####################................
        K1 = xor_bytes(K1, Rb)
    
    K2 = LEFT_shift(K1)
    if K1[0] & 0x80:
        K2 = xor_bytes(K2, Rb)

    #>>>>>>>>>>>>>>>> padding the plain text 
    n = len(message)
    if n % block_size == 0:
        last_block = xor_bytes(message[-block_size:], K1)
    else:
        
        padded = message + b'\x80' + b'\x00' * (block_size - 1 - (n % block_size))
        last_block = xor_bytes(padded[-block_size:], K2)

    #>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>> calculate cmac 
    c = b'\x00' * block_size
    for i in range(0, len(message) - block_size, block_size):
        block = message[i:i+block_size]
        c = xor_bytes(c, block)
        c = encryptor.update(c) + encryptor.finalize()   #######>>>>>>>>> this wont work 
################################################>>>>>>>>>>>>>>>
    ###>>>>>>>>>>>>>>>>>>>>>>>>>>>>> Last block enc
    c = xor_bytes(c, last_block)
    final = encryptor.update(c) + encryptor.finalize()

    return final.hex()


####--------------------............TEST.........----------------------------------

key = b"free_internetttt"
msg = b"freddy fazbear and kanye west are same person"
print("Manual CMAC-AES:", cmac_aes_manual(key, msg))

AlreadyFinalized: Context was already finalized.

# CMAC Implementation 2 
the good way

In [4]:
from Crypto.Hash import CMAC
from Crypto.Cipher import AES

#####>>>>>....... insert key (16 or 24 or 32 bits)
key = b"meow_meow_meow67"  ### this one is 16  

###>>>>..... cmac method 
cmac = CMAC.new(key, ciphermod=AES)
####>>>>>>........ plain txt
cmac.update(b"the weather outside is rizzy")

print("CMAC with AES encrypted result :")
print(cmac.hexdigest())
print(cmac.digest)

CMAC with AES encrypted result :
67415a1985f74ec24c2de9cb4406d721
<bound method CMAC.digest of <Crypto.Hash.CMAC.CMAC object at 0x000002EA0FE80190>>
